# Optical Cockpit — Energy Accounting

**Stage:** 8B — Optical Cockpit + Energy Accounting  
**Model status:** `energy_accounting_prediction` / `exposure_bookkeeping`  
**Final export allowed:** False

---

## What this notebook does

This notebook performs **energy and exposure bookkeeping only**.
It does not compute material damage, modification, ablation, Δn, voids, cracks, or waveguide formation.

Outputs:
1. Energy ledger (component-by-component throughput)
2. Energy at sample
3. Average power before and after optics
4. Focus fluence estimate from effective beam area
5. Peak intensity estimate (approximate — no nonlinear corrections)
6. Pulse spacing
7. Effective pulses per spot
8. Dose-per-unit-length proxy
9. Static multi-pulse exposure
10. Caveat panel

**Stage 8C** will connect this cockpit to real optical field arrays from `bessel_twin_core`.

In [1]:
# ============================================================
# USER CONTROLS — edit this cell to adjust the cockpit
# ============================================================

planning_mode = True           # True = outputs are planning estimates only
save_outputs = False           # True = save CSV and figure (requires show_caveats=True)
figure_dpi = 180
show_caveats = True            # Must be True to save outputs
show_diagnostic_panels = True

# --- Laser source ---
wavelength_nm = 1030.0
pulse_duration_fs = 260.0
repetition_rate_Hz = 25_000.0
pulse_energy_before_optics_uJ = 200.0
average_power_limit_W = 10.0

# --- Beam geometry ---
beam_radius_mm = 2.0
effective_area_um2 = 100.0       # Approximate beam area at focus for fluence estimate

# --- Optical chain efficiencies ---
slm_diffraction_efficiency = 0.75
selected_first_order_fraction = 0.73
relay_transmission = 0.90
objective_transmission = 0.85
sample_interface_transmission = 0.95

# --- Writing geometry ---
scan_speed_mm_s = 1.0
line_length_um = 500.0
effective_diameter_um = 3.0
num_static_pulses = 100

In [2]:
# Save guard — raise if trying to save with caveats hidden
if save_outputs and not show_caveats:
    raise ValueError(
        "Cannot save outputs with show_caveats=False. "
        "Energy and exposure bookkeeping outputs must always carry caveats. "
        "Set show_caveats=True before saving."
    )

In [3]:
from pathlib import Path
import sys
import numpy as np

# Ensure vbb_study is importable
_repo_root = Path.cwd()
while _repo_root.name and not (_repo_root / 'vbb_study').is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from vbb_study.digital_twin.energy_accounting import (
    average_power_w,
    compute_energy_ledger,
    fluence_from_effective_area_j_cm2,
    peak_intensity_w_cm2,
    default_holographic_chain,
)
from vbb_study.digital_twin.exposure_bookkeeping import (
    pulse_spacing_um,
    pulses_per_spot,
    dose_per_unit_length_proxy,
    static_exposure_total_energy_uJ,
    line_exposure_summary,
)

print("Modules loaded. Running energy and exposure bookkeeping (Stage 8B).")

Modules loaded. Running energy and exposure bookkeeping (Stage 8B).


In [4]:
# ============================================================
# 1. BUILD OPTICAL CHAIN AND COMPUTE ENERGY LEDGER
# ============================================================

chain = default_holographic_chain(
    slm_diffraction_efficiency=slm_diffraction_efficiency,
    selected_first_order_fraction=selected_first_order_fraction,
    relay_transmission=relay_transmission,
    objective_transmission=objective_transmission,
    sample_interface_transmission=sample_interface_transmission,
)

ledger = compute_energy_ledger(
    pulse_energy_before_optics_uJ=pulse_energy_before_optics_uJ,
    repetition_rate_Hz=repetition_rate_Hz,
    components=chain,
    average_power_limit_W=average_power_limit_W,
)

print(f"Input pulse energy:    {pulse_energy_before_optics_uJ:.2f} µJ")
print(f"Energy at sample:      {ledger.energy_at_sample_uJ:.4f} µJ")
print(f"Total throughput:      {ledger.total_throughput_fraction:.3%}")
print(f"Average power (input): {average_power_w(pulse_energy_before_optics_uJ, repetition_rate_Hz)*1000:.2f} mW")
print(f"Average power (sample):{ledger.average_power_at_sample_W*1000:.2f} mW")
if ledger.ledger_warnings:
    for w in ledger.ledger_warnings:
        print(f"  ⚠ WARNING: {w}")

Input pulse energy:    200.00 µJ
Energy at sample:      79.5791 µJ
Total throughput:      39.790%
Average power (input): 5000.00 mW
Average power (sample):1989.48 mW


In [5]:
# ============================================================
# 2. ENERGY LEDGER TABLE
# ============================================================

print(f"{'Component':<35} {'Frac':>7} {'Cumul':>7} {'E_in µJ':>10} {'E_out µJ':>10} {'P_out mW':>10}")
print('-' * 85)
for row in ledger.rows:
    status = '' if row.enabled else '[DISABLED]'
    print(
        f"{row.component_name:<35} {row.fraction_this_component:>7.3f} "
        f"{row.cumulative_fraction:>7.3f} {row.energy_in_uJ:>10.4f} "
        f"{row.energy_out_uJ:>10.4f} {row.average_power_out_W*1000:>10.3f} {status}"
    )
print('-' * 85)
print(f"{'SAMPLE':35} {'':7} {ledger.total_throughput_fraction:>7.3f} "
      f"{'':10} {ledger.energy_at_sample_uJ:>10.4f} {ledger.average_power_at_sample_W*1000:>10.3f}")
print(f"\nModel status: {ledger.model_status} | Final export allowed: {ledger.final_export_allowed}")

Component                              Frac   Cumul    E_in µJ   E_out µJ   P_out mW
-------------------------------------------------------------------------------------
pre_slm_optics                        1.000   1.000   200.0000   200.0000   5000.000 
slm_diffraction_efficiency            0.750   0.750   200.0000   150.0000   3750.000 
selected_first_order_fraction         0.730   0.547   150.0000   109.5000   2737.500 
relay_optics                          0.900   0.493   109.5000    98.5500   2463.750 
objective_transmission                0.850   0.419    98.5500    83.7675   2094.188 
sample_interface_transmission         0.950   0.398    83.7675    79.5791   1989.478 
additional_user_loss                  1.000   0.398    79.5791    79.5791   1989.478 
-------------------------------------------------------------------------------------
SAMPLE                                        0.398               79.5791   1989.478

Model status: energy_accounting_prediction | Final expo

In [6]:
# ============================================================
# 3. FOCUS FLUENCE ESTIMATE (effective area mode)
# ============================================================

fluence_est = fluence_from_effective_area_j_cm2(
    pulse_energy_uJ=ledger.energy_at_sample_uJ,
    effective_area_um2=effective_area_um2,
)

print(f"Focus fluence estimate (effective area mode):")
print(f"  Pulse energy at sample: {ledger.energy_at_sample_uJ:.4f} µJ")
print(f"  Effective beam area:    {effective_area_um2:.1f} µm²")
print(f"  Estimated peak fluence: {fluence_est:.4f} J/cm²")
print(f"  Model status: fluence_prediction (approximate — uses assumed effective area)")
print()
print("NOTE: This is an order-of-magnitude estimate.")
print("Stage 8C will replace this with real optical field arrays from bessel_twin_core.")

Focus fluence estimate (effective area mode):
  Pulse energy at sample: 79.5791 µJ
  Effective beam area:    100.0 µm²
  Estimated peak fluence: 79.5791 J/cm²
  Model status: fluence_prediction (approximate — uses assumed effective area)

NOTE: This is an order-of-magnitude estimate.
Stage 8C will replace this with real optical field arrays from bessel_twin_core.


In [7]:
# ============================================================
# 4. PEAK INTENSITY ESTIMATE
# ============================================================

intensity_est = peak_intensity_w_cm2(
    peak_fluence_j_cm2_val=fluence_est,
    pulse_duration_fs=pulse_duration_fs,
    temporal_shape="flat_top_equivalent",
)

print(f"Peak intensity estimate:")
print(f"  Peak fluence:    {intensity_est.peak_fluence_j_cm2:.4f} J/cm²")
print(f"  Pulse duration:  {intensity_est.pulse_duration_fs:.1f} fs")
print(f"  Temporal shape:  {intensity_est.temporal_shape}")
print(f"  Peak intensity:  {intensity_est.peak_intensity_w_cm2:.3e} W/cm²")
print()
print(f"  Approximation: {intensity_est.approximation_note}")
print(f"  Model status: {intensity_est.model_status}")

Peak intensity estimate:
  Peak fluence:    79.5791 J/cm²
  Pulse duration:  260.0 fs
  Temporal shape:  flat_top_equivalent
  Peak intensity:  3.061e+14 W/cm²

  Approximation: Flat-top equivalent: I_peak = F_peak / tau. Conservative approximation. Peak intensity estimate assumes no nonlinear reshaping and no plasma/thermal feedback.
  Model status: energy_accounting_prediction


In [8]:
# ============================================================
# 5. EXPOSURE BOOKKEEPING
# ============================================================

ds = pulse_spacing_um(scan_speed_mm_s, repetition_rate_Hz)
n_eff = pulses_per_spot(effective_diameter_um, scan_speed_mm_s, repetition_rate_Hz)
d_line = dose_per_unit_length_proxy(ledger.energy_at_sample_uJ, repetition_rate_Hz, scan_speed_mm_s)
e_static = static_exposure_total_energy_uJ(ledger.energy_at_sample_uJ, num_static_pulses)

print(f"Exposure bookkeeping (model status: exposure_bookkeeping):")
print(f"  Scan speed:              {scan_speed_mm_s:.2f} mm/s")
print(f"  Repetition rate:         {repetition_rate_Hz:.0f} Hz")
print(f"  Pulse spacing:           {ds:.4f} µm")
print(f"  Effective diameter:      {effective_diameter_um:.2f} µm")
print(f"  Pulses per spot (N_eff): {n_eff:.2f}")
print(f"  Dose/length proxy:       {d_line:.4f} J/m  [not calibrated material dose]")
print(f"  Static exposure ({num_static_pulses} pulses): {e_static:.4f} µJ total")

Exposure bookkeeping (model status: exposure_bookkeeping):
  Scan speed:              1.00 mm/s
  Repetition rate:         25000 Hz
  Pulse spacing:           0.0400 µm
  Effective diameter:      3.00 µm
  Pulses per spot (N_eff): 75.00
  Dose/length proxy:       1989.4781 J/m  [not calibrated material dose]
  Static exposure (100 pulses): 7957.9125 µJ total


In [9]:
# ============================================================
# 6. LINE EXPOSURE SUMMARY
# ============================================================

summary = line_exposure_summary(
    pulse_energy_at_sample_uJ=ledger.energy_at_sample_uJ,
    repetition_rate_Hz=repetition_rate_Hz,
    scan_speed_mm_s=scan_speed_mm_s,
    line_length_um=line_length_um,
    effective_diameter_um=effective_diameter_um,
)

print(f"Line exposure summary ({line_length_um:.0f} µm line):")
print(f"  Pulse spacing:           {summary['pulse_spacing_um']:.4f} µm")
print(f"  Pulses per spot:         {summary['pulses_per_spot']:.2f}")
print(f"  Overlap fraction:        {summary['overlap_fraction']:.3f}")
print(f"  Line scan duration:      {summary['line_duration_s']*1000:.3f} ms")
print(f"  Total pulses on line:    {summary['total_pulses_on_line']}")
print(f"  Total energy on line:    {summary['total_energy_on_line_uJ']:.4f} µJ")
print(f"  Dose/length proxy:       {summary['dose_per_unit_length_J_m']:.4f} J/m")
if summary['warnings']:
    for w in summary['warnings']:
        print(f"  ⚠ WARNING: {w}")

Line exposure summary (500 µm line):
  Pulse spacing:           0.0400 µm
  Pulses per spot:         75.00
  Overlap fraction:        0.987
  Line scan duration:      500.000 ms
  Total pulses on line:    12500
  Total energy on line:    994739.0625 µJ
  Dose/length proxy:       1989.4781 J/m


In [ ]:
# ============================================================
# 7. DIAGNOSTIC FIGURE (if show_diagnostic_panels=True)
# ============================================================

if show_diagnostic_panels:
    import matplotlib
    matplotlib.use('Agg')  # safe for non-interactive/CI use
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Left: energy through chain
    ax = axes[0]
    labels = [r.component_name.replace('_', '\n') for r in ledger.rows if r.enabled]
    energies = [r.energy_out_uJ for r in ledger.rows if r.enabled]
    ax.bar(range(len(labels)), energies, color='steelblue')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('Pulse energy (µJ)')
    ax.set_title('Energy through chain')
    ax.axhline(ledger.energy_at_sample_uJ, color='red', ls='--', lw=1,
               label=f'Sample: {ledger.energy_at_sample_uJ:.3f} µJ')
    ax.legend(fontsize=7)

    # Middle: average power through chain
    ax = axes[1]
    powers_mW = [r.average_power_out_W * 1000 for r in ledger.rows if r.enabled]
    ax.bar(range(len(labels)), powers_mW, color='darkorange')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('Average power (mW)')
    ax.set_title('Average power through chain')
    if average_power_limit_W is not None:
        ax.axhline(average_power_limit_W * 1000, color='red', ls='--', lw=1,
                   label=f'Limit: {average_power_limit_W*1000:.0f} mW')
        ax.legend(fontsize=7)

    # Right: exposure bookkeeping summary
    ax = axes[2]
    ax.axis('off')
    summary_text = (
        f"EXPOSURE BOOKKEEPING SUMMARY\n"
        f"────────────────────────────\n"
        f"E at sample:  {ledger.energy_at_sample_uJ:.4f} µJ\n"
        f"Fluence est:  {fluence_est:.4f} J/cm²\n"
        f"Peak I est:   {intensity_est.peak_intensity_w_cm2:.2e} W/cm²\n"
        f"Pulse spacing: {ds:.3f} µm\n"
        f"N_eff:        {n_eff:.1f}\n"
        f"Dose/length:  {d_line:.4f} J/m\n"
    )
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top', fontfamily='monospace')

    caveat = (
        "DIAGNOSTIC ONLY — model_status=energy_accounting_prediction — "
        "final_export_allowed=False\n"
        "This figure does not predict material modification, damage, or waveguide formation."
    )
    fig.text(0.5, -0.02, caveat, ha='center', fontsize=7, color='red',
             style='italic', wrap=True)

    fig.suptitle(
        f'Stage 8B — Optical Cockpit Energy Ledger\n'
        f'λ={wavelength_nm:.0f} nm  τ={pulse_duration_fs:.0f} fs  '
        f'f={repetition_rate_Hz:.0f} Hz  E_in={pulse_energy_before_optics_uJ:.0f} µJ',
        fontsize=10
    )
    plt.tight_layout()

    if save_outputs:
        from pathlib import Path
        out_dir = Path('outputs/figures/digital_twin')
        out_dir.mkdir(parents=True, exist_ok=True)
        fig_path = out_dir / 'stage8b_energy_ledger_preview.png'
        fig.savefig(fig_path, dpi=figure_dpi, bbox_inches='tight')
        print(f"Saved: {fig_path}")
        print("Metadata: stage=stage8b_optical_cockpit_energy_accounting")
        print("          figure_status=diagnostic_allowed")
        print("          model_status=energy_accounting_prediction")
        print("          final_export_allowed=False")
    else:
        plt.show()
    plt.close(fig)

C:\Users\sm2006\AppData\Local\Temp\ipykernel_10076\2514478281.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


: 

## Caveats

**This notebook performs energy and exposure bookkeeping only.**
It does not compute material damage, modification, ablation, Δn, voids, cracks, or waveguide formation.

### Claim boundaries at Stage 8B

| Output | Allowed claim | Not allowed |
|---|---|---|
| Energy at sample | Energy delivered to sample focal region | Energy absorbed by material |
| Fluence estimate | Approximate peak fluence given effective area | Fluence that modifies material |
| Peak intensity estimate | Approximate peak intensity (no nonlinear effects) | Actual intensity inside material |
| Pulse spacing | Geometric distance between consecutive spots | Track continuity or modification |
| Pulses per spot | Geometric number of overlapping pulses | Multi-shot modification dose |
| Dose-per-length | Proxy E×f/v — not a calibrated material dose | Material dose or absorbed energy |

### What Stage 8C will add

Stage 8C will connect this cockpit to real optical-field arrays from `bessel_twin_core`:
- The effective area estimate will be replaced by `∫|U(x,y)|² dA` from the actual field.
- Fluence maps will be computed from `F(x,y) = E_sample × |U(x,y)|² / ∫|U|² dA`.
- The energy ledger output will be wired to the `scale_intensity_to_fluence_j_cm2` function.

### Model status

All outputs in this notebook carry model status `energy_accounting_prediction` or
`exposure_bookkeeping`.  No material-response model is implemented.  No calibrated
threshold is applied.  No modification, damage, or feature-formation claim is made.